# ViT & CLIP- When Transformers Learned to See

**Module 6 · Lesson 6.4**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Vision Transformer - Images as Tokens
- CLIP - Two Encoders, One Shared Space
- Contrastive Learning - How CLIP Trains
- Zero-Shot Classification - No Labels Needed
- Using CLIP - Search, Conditioning, and Limits
- The 2026 Landscape and Production

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install openai torch -q

# Reproducibility - the lesson uses seeded random values throughout

In [ ]:
# Download a sample image used by the CLIP / SigLIP demos below.
import urllib.request, os
if not os.path.exists("dog.jpg"):
    urllib.request.urlretrieve(
        "https://github.com/pytorch/hub/raw/master/images/dog.jpg", "dog.jpg")
print("dog.jpg ready")

## The whole game: one shared space for pictures and words

> 🗺️ **Analogy**
>
> **Picture a universal translator that turns both photos and sentences into "meaning coordinates" on one map.** Feed it a picture of a dog or the caption "a dog" and it drops a pin at almost the same spot - because they mean the same thing. How close two pins are (the angle between them - the *cosine similarity*) is how well the picture and the words match. A **Vision Transformer (ViT)** is the part that reads a picture into those coordinates; a text encoder does it for words; and **CLIP** is the pair, trained so matched pairs land together.
> That one idea - images and text in the **same** space - is what makes zero-shot classification, image search, and text-to-image conditioning possible. You already used it: it is the text encoder that steered Stable Diffusion (6.2) and the image features behind IP-Adapter (6.3).
> **Where the analogy breaks down:** a real translator understands grammar and word order. CLIP is closer to a bag-of-concepts matcher - it nails "dog + beach + sunset" but fumbles "the dog to the *left* of the cat", counting, and reading text in an image, because it crushes everything into one blurry vector. Those are the failure modes in Step 5.

This builds on Module 4 (the transformer and attention - a ViT is just that transformer run over image patches) and on Lessons 4.1 / 6.1 (embeddings as vectors). It pays off the "CLIP is opened up in 6.4" promise from 6.2 and 6.3. One forward hook: multimodal LLMs (Lesson 6.5) use a CLIP-style encoder as their "eyes".

- **Explain** how a Vision Transformer turns an image into a sequence of patch tokens and encodes it into a single embedding, reusing the transformer from Module 4.

- **Describe** how CLIP places images and text in one shared embedding space, and how cosine similarity there measures an image-text match.

- **Apply** a pretrained CLIP model to zero-shot classification and image-text search with `transformers`, and read the similarity scores.

- **Compare** the 2026 encoder landscape - CLIP vs SigLIP 2 vs DINOv3 - and choose the right one for retrieval, zero-shot, or dense/spatial tasks.

## Warm-up: 60 seconds of recall

Tap each card to check yourself. Each one is load-bearing for today.

---

## 🎯 Concept 1: Vision Transformer - Images as Tokens

### Vision Transformer - Images as Tokens

Cut the image into patches, treat each patch as a token, and run the transformer from Module 4.

**Reading a comic strip panel by panel.** You do not take in the whole page as one blur - you scan it as a sequence of panels, and your brain relates them. A ViT chops the image into panels (patches) and lets attention relate them, the same way it relates words in a sentence.

The gain: a ViT = a transformer over image patches. Image -> patches -> tokens -> one embedding. A "token" here is just one patch turned into a vector.

**📝 `patchify.py`** - *PyTorch*

In [ ]:
import torch, torch.nn as nn

image = torch.randn(1, 3, 224, 224)          # one RGB image (batch of 1)

# 1) cut into 16x16 patches with a strided conv - each patch -> a 768-dim vector
patchify = nn.Conv2d(3, 768, kernel_size=16, stride=16)
tokens = patchify(image).flatten(2).transpose(1, 2)  # (1, 196, 768)
print("patch tokens:", tuple(tokens.shape))     # (224/16)^2 = 196 tokens
# Output: patch tokens: (1, 196, 768)

# 2) prepend a [CLS] token, add position embeddings, run a transformer layer
cls = nn.Parameter(torch.randn(1, 1, 768))       # nn.Parameter = a tensor the optimizer trains (starts random, learned)
pos = nn.Parameter(torch.randn(1, 197, 768))     # one position vector per token
seq = torch.cat([cls, tokens], dim=1) + pos       # (1, 197, 768)

encoder = nn.TransformerEncoderLayer(768, nhead=12, batch_first=True)  # nhead=12: 12 parallel attention heads (Module 4)
image_embedding = encoder(seq)[:, 0]              # the [CLS] row = one image vector
print("image embedding:", tuple(image_embedding.shape))
# Output: image embedding: (1, 768)

- **Patchify** - a strided `Conv2d` with `kernel=stride=16` is a neat trick: it slides a 16x16 window with no overlap, so each window (patch) becomes one 768-dim vector. The conv outputs `(1, 768, 14, 14)`; `flatten(2)` collapses the 14x14 grid into 196, giving `(1, 768, 196)`; `transpose(1, 2)` swaps to `(1, 196, 768)` - 196 tokens, each a 768-dim vector.

- **The `[CLS]` token** - a learned extra token prepended to the sequence; after the transformer, its output row is used as the single "whole image" embedding (a common summarizing trick).

- **Position embeddings** - a bag of patches has no order, so each token gets a learned position vector added; that is how the model recovers *where* each patch sat.

- **The transformer** is the same `TransformerEncoderLayer` from Module 4 - self-attention lets every patch look at every other. The pooled output is one image embedding, the vector CLIP compares against text next. (Real ViTs stack ~12 such layers; ViT-B/16 is ~86M parameters.)

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?
  A Vision Transformer turns an image into a **sequence of patch tokens**: cut into fixed patches, flatten each to a vector, add a position embedding, and run the Module-4 transformer over the sequence. The pooled output (often a `[CLS]` token) is one **image embedding**. No convolutions-as-the-whole-model, no built-in locality - just attention over patches. That single vector is what CLIP will compare against text.

---

## 🎯 Concept 2: CLIP - Two Encoders, One Shared Space

### CLIP - Two Encoders, One Shared Space

An image encoder and a text encoder trained to put a picture and its caption at the same point.

**Two people describing the same city on the same map.** One draws from a photo, one from a written description - but if they agree to use the *same* map with the same landmarks, their two pins land in the same place. CLIP trains the picture-reader and the word-reader to use one shared map.

The gain: CLIP = image encoder + text encoder into one shared space. Compare with cosine similarity: 1 = same direction (match), 0 = unrelated.

**📝 `clip_similarity.py`** - *transformers*

In [ ]:
# pip install transformers pillow torch  (runs on CPU; no GPU needed)
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import torch

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

image = Image.open("dog.jpg")
texts = ["a photo of a dog", "a photo of a cat", "a photo of a car"]

# the processor resizes/normalizes the image and tokenizes the text
inputs = proc(text=texts, images=image, return_tensors="pt", padding=True)
with torch.no_grad():
    out = model(**inputs)

# logits_per_image = cosine similarity (scaled); softmax -> a prob over the 3 texts
probs = out.logits_per_image.softmax(dim=1)[0]
for t, p in zip(texts, probs):
    print(f"{p:.2f}  {t}")
# Output: 0.97  a photo of a dog
# Output: 0.02  a photo of a cat
# Output: 0.01  a photo of a car

- `CLIPModel` bundles both encoders; `CLIPProcessor` does the image preprocessing and text tokenization so you do not have to. It runs fine on CPU - no `.to("cuda")` needed for one image.

- One forward pass embeds the image and all three captions into the shared space and returns `logits_per_image` - the raw (pre-softmax) match scores. These are the cosine similarity times CLIP's large learned temperature (`logit_scale`, ~100); that scaling is what lets softmax turn small raw cosine gaps into a sharply peaked 0.97, and it is why you cannot read the raw number as a cosine.

- `.softmax(dim=1)` turns those scores into a probability *over the given captions* (`dim=1` normalizes across the captions, so they compete and the three numbers sum to 1). It is relative to the three you supplied - not a calibrated "how doggy" score (Step 4's anti-example).

- To get the raw vectors instead, call `model.get_image_features(...)` / `get_text_features(...)` and compare them yourself - that is what Step 5's search does.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?
  CLIP is **two encoders trained into one shared space**: a ViT for images, a transformer for text. In that space, **cosine similarity** measures how well a picture and a caption match. You embed the image and any candidate texts and compare - which is the engine under zero-shot classification, search, and the text conditioning you already used in 6.2/6.3.

---

## 🎯 Concept 3: Contrastive Learning - How CLIP Trains

### Contrastive Learning - How CLIP Trains

Score every image against every caption in a batch; pull the correct pairs together, push the rest apart.

**Matching name tags to faces at a party.** Lay out N photos and N name tags; the game is to pair each photo with its own tag and no one else's. Play it millions of times and you get very good at reading which face goes with which name. CLIP plays exactly this game with images and captions.

The gain: contrastive = pull matched pairs together, push mismatches apart. The correct pairs are the diagonal of an image x caption grid.

**📝 `contrastive.py`** - *PyTorch*

In [ ]:
import torch, torch.nn.functional as F

# A toy batch: 4 image embeddings and their 4 matched caption embeddings.
# (normalize so a dot product IS the cosine similarity)
img_emb = F.normalize(torch.randn(4, 512), dim=1)   # 4 images
txt_emb = F.normalize(torch.randn(4, 512), dim=1)   # their 4 captions

logits = img_emb @ txt_emb.T                        # (4, 4) image-vs-caption grid

# the CORRECT pairs are the diagonal: image i matches caption i
labels = torch.arange(4)                            # [0, 1, 2, 3]
# symmetric cross-entropy: each row picks its caption, each column picks its image
loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
print("contrastive loss:", round(loss.item(), 2))
# Output: contrastive loss: 1.39   (illustrative - with random init it starts near
# Output: log(4)=1.39 or above and falls toward 0 as the diagonal wins; varies by seed)

- `img_emb @ txt_emb.T` is the NxN grid: entry `(i, j)` is the cosine similarity of image *i* to caption *j*. The **diagonal** is the correct pairs.

- `labels = arange(N)` says "row i should pick column i". `cross_entropy` on the rows pushes each image toward its own caption and away from the others - and again on the columns (`.T`) for symmetry.

- As training runs, the diagonal similarities climb and the off-diagonal fall - the loss drops. The **shared space from Step 2 is a by-product** of winning this game.

- This uses random vectors, so the printed loss is illustrative (it varies by seed); the point is the shape of the objective, not the exact number.

> 📦 **Info**
>
> 🔄 Softmax (CLIP) vs sigmoid (SigLIP) - the 2026 shift
> CLIP's loss is a **softmax** over each row: the correct caption must *win* against the others, so it needs very large batches full of negatives. **SigLIP** (Google) swaps to a **sigmoid** loss: each image-caption cell is judged on its own as match (1) or not (0), with no row-wise competition. No normalization across the batch means no dependence on giant batches - simpler and it scales, which is why the 2026 encoders (SigLIP 2) use it. The animation's scene 2 shows the difference.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?
  CLIP trains with **contrastive learning** and no labels: score every image against every caption in a batch, and push each image toward its own caption (the diagonal) and away from the rest. The shared space falls out of that game. **SigLIP**'s sigmoid loss judges each pair independently, dropping CLIP's big-batch dependence - the training recipe behind the 2026 encoders.

---

## 🎯 Concept 4: Zero-Shot Classification - No Labels Needed

### Zero-Shot Classification - No Labels Needed

Classify into any set of classes you can name, without training a classifier - just compare to class prompts.

**A multiple-choice test where you write the choices.** You hold up a photo and offer the model a few captions to pick from ("a photo of a dog", "a photo of a cat", ...). It picks the closest. Change the choices and you have a new classifier - instantly, no retraining.

The gain: zero-shot = classify by nearest class prompt. "Zero-shot" means no labeled training set - not zero assumptions.

**📝 `zero_shot.py`** - *transformers*

In [ ]:
# reuse `model`, `proc`, and `image` from Step 2
classes = ["dog", "cat", "car", "pizza"]
prompts = [f"a photo of a {c}" for c in classes]   # prompt template matters

inputs = proc(text=prompts, images=image, return_tensors="pt", padding=True)
with torch.no_grad():
    probs = model(**inputs).logits_per_image.softmax(dim=1)[0]

best = probs.argmax().item()
print(f"predicted: {classes[best]}  ({probs[best]:.2f})")
# Output: predicted: dog  (0.96)

- Build one text prompt per class with a **template** (`"a photo of a {}"`), embed the image against all of them, and take the `argmax` similarity. That is the entire classifier - no training loop, no labels.

- Swap `classes` for any label set (dog breeds, product types, defect vs no-defect) and you have a new classifier instantly - CLIP's headline capability.

- Averaging several templates ("a photo of a {}", "a close-up of a {}", "a blurry photo of a {}") and their embeddings ("prompt ensembling") typically nudges accuracy up further.

- The returned probability is *relative to your class set* - add "a photo of a hot dog" and the numbers shift. It is a ranking over your prompts, not a calibrated confidence.

#### 💡 What just happened

⚡ What just happened?
**Zero-shot classification** turns CLIP's shared space into a classifier for *any* classes you can name: embed the image, embed each class as a prompt (`"a photo of a {}"`), take the nearest. No labeled training set. The prompt *wording* is a real lever, and the score is a ranking over your prompts - not a calibrated probability.

---

## 🎯 Concept 5: Using CLIP - Search, Conditioning, and Limits

### Using CLIP - Search, Conditioning, and Limits

The same shared space powers image search, the SD text encoder, and IP-Adapter - and it has real blind spots.

**A library where every book and every question live at a coordinate.** Ask a question, go to its coordinate, and grab the nearest books. CLIP indexes images at coordinates and sends your text query to the same map to find neighbours - search by meaning, not keywords.

The gain: one shared space -> search, conditioning, and moderation all reuse it. But one blurry vector can't count or place things.

**📝 `image_search.py`** - *transformers*

> 📝 **image_search.py** — illustrative (requires a `product_photos/` gallery of images on disk; shown for reading, not executed on Restart & Run All).

```python
import torch, torch.nn.functional as F
from PIL import Image

# INDEX ONCE: store a normalized CLIP embedding per image.
from glob import glob
image_paths = sorted(glob("product_photos/*.jpg"))   # your image folder

def embed_image(path):
    inp = proc(images=Image.open(path), return_tensors="pt")
    with torch.no_grad():
        return F.normalize(model.get_image_features(**inp), dim=1)

gallery = torch.cat([embed_image(p) for p in image_paths])   # (N, 512)

# SEARCH WITH TEXT: embed the query, rank images by cosine similarity.
def search(query, k=3):
    inp = proc(text=[query], return_tensors="pt", padding=True)
    with torch.no_grad():
        q = F.normalize(model.get_text_features(**inp), dim=1)
    scores = (gallery @ q.T).squeeze(1)            # cosine sim to every image
    return scores.topk(min(k, gallery.size(0))).indices.tolist()

print(search("a red dress with polka dots"))
# Output: [42, 7, 118]   (indices of the 3 closest images)
```

- **Index once**: embed every image and store the vectors (in production, in a vector database - FAISS/pgvector/Qdrant - which does the nearest-neighbour search at scale; the depth is Module 8).

- **Search**: embed the text query into the *same* space and rank images by cosine similarity. Same trick, query side is text instead of an image - because both live in one space.

- **You have already used this space:** the CLIP *text* encoder is what turned your prompt into guidance vectors for Stable Diffusion (6.2), and CLIP *image* features are what IP-Adapter injects (6.3). This step is the "what was inside".

- **Content moderation** is the same pattern - score images against unsafe-concept prompts - which is powerful but inherits CLIP's biases and errors, so it needs human review (ethics depth in Module 15).

> 📦 **Info**
>
> 🚫 CLIP's blind spots (know them before you ship)
> - **Counting** - "three cats" vs "two cats" often look the same to CLIP; the vector encodes "cat-ness", not quantity.
> - **Spatial relations** - "the cup to the left of the plate" vs "right of" collapse together; position is largely lost.
> - **Text in images** - it does not reliably read words rendered in a picture (that is OCR's job).
> - **Fine-grained & compositional** - subtle species/parts and multi-attribute prompts ("a red cube on a blue sphere") trip it up.
> - **Bias** - it inherits the associations in its web training data; audit before using it for moderation or ranking (Module 15).

#### 💡 What just happened

⚡ What just happened?
  The shared space is a general tool. **Image search** embeds a gallery once and ranks by cosine similarity to a text query; the same space is CLIP's role as the **SD text encoder** (6.2) and **IP-Adapter** features (6.3), and the basis of **content moderation**. But one blurry vector can't count, place, or read text in images - know those limits before you trust it.

---

## 🎯 Concept 6: The 2026 Landscape and Production

### The 2026 Landscape and Production

CLIP is the mental model, not the frontier: SigLIP 2, DINOv3, and the practical stack.

**Different lenses for different jobs.** A wide-angle lens (text-aligned CLIP/SigLIP) is great for "does this picture match these words?"; a macro lens (DINO) is great for fine, dense detail with no words at all. You pick the lens for the shot, not the newest body for everything.

The gain: match the encoder to the task - retrieval/zero-shot vs dense/spatial. CLIP is the mental model; SigLIP 2 / DINOv3 are the 2026 tools.

**📝 `siglip2.py`** - *transformers*

In [ ]:
# 2026: SigLIP 2 - the strong open successor to CLIP (sigmoid loss, multilingual).
from transformers import AutoModel, AutoProcessor
from PIL import Image
import torch

model = AutoModel.from_pretrained("google/siglip2-base-patch16-224")
proc  = AutoProcessor.from_pretrained("google/siglip2-base-patch16-224")

inputs = proc(text=["a photo of a dog", "a photo of a cat"],
              images=Image.open("dog.jpg"), return_tensors="pt", padding="max_length")
with torch.no_grad():
    logits = model(**inputs).logits_per_image

# SigLIP uses a SIGMOID: each score is an independent match probability, not a
# softmax over the row - so the two numbers need not sum to 1.
probs = torch.sigmoid(logits)[0]
print([round(p.item(), 2) for p in probs])
# Output: [0.19, 0.00]   (dog is the match; both are low - sigmoid is absolute, not a softmax that sums to 1)

- SigLIP 2 uses the same `AutoModel` shape as CLIP, so switching encoders is a one-line change - but note the `torch.sigmoid` (independent per-pair scores) instead of a softmax, matching Step 3's loss.

- **Choose by task:** text-image retrieval / zero-shot -> CLIP or (better, multilingual) SigLIP 2; dense/spatial features (segmentation, depth, no text) -> DINOv3 (*self-supervised* = learns from images alone, no captions); promptable segmentation (you click or box a region and it cuts the object out) -> SAM 2.

- **The stack:** `transformers` (CLIP/SigLIP), `open_clip` (the LAION-trained model zoo), and `timm` (the backbone library) - plus a vector database for production search/moderation.

- Everything here is a pretrained *encoder*; for a fixed task you can fine-tune or linear-probe it (Module 9), and multimodal LLMs (Lesson 6.5) bolt a CLIP-style encoder onto a language model as its "eyes".

| Encoder | Trained with | Best at | When to reach for it |
|---|---|---|---|
| CLIP(OpenAI, 2021) | Softmax contrastive, ~400M pairs | The mental model; solid zero-shot | Learning, quick baselines, English |
| SigLIP 2(Google, 2025) | Sigmoid loss, 40B+ pairs, multilingual | Retrieval, zero-shot, dense features | The default open text-image encoder |
| DINOv3(Meta, 2025) | Self-supervised (no text) | Dense/spatial features, segmentation | When you need pixels-and-where, no text |
| SAM 2(Meta) | Promptable segmentation | Segment anything, in images/video | Cutting objects out, masks |

#### 💡 What just happened

⚡ What just happened?
  The original **CLIP is the teaching baseline, not the 2026 frontier**. **SigLIP 2** (sigmoid loss, multilingual, better features) is the go-to open text-image encoder; **DINOv3** leads self-supervised dense/spatial features; **SAM 2** segments. The stack is `transformers` / `open_clip` / `timm` plus a vector database. Match the encoder to the task.

## Synthesis: one shared space, many uses

### The complete picture, in one breath

A **Vision Transformer** reads an image as a sequence of patch tokens and pools it into one **embedding**; a text transformer does the same for words. **CLIP** trains the two together with a **contrastive** objective so a picture and its caption land at the same point in one **shared space**, where **cosine similarity** measures match. That one space does everything: **zero-shot classification** (nearest class prompt), **image search** (nearest image to a text query), the **SD text encoder** (6.2) and **IP-Adapter** (6.3) you already used, and content moderation - within its limits (no counting, spatial, or in-image text). In 2026 you reach for **SigLIP 2** for retrieval and **DINOv3** for dense features, but the idea you learned - images and words in one space - is unchanged.

> ℹ️ **Info**
>
> Where this goes next in Module 6
> - **Lesson 6.5 - Multimodal Models:** LLaVA-style multimodal LLMs bolt a CLIP-style encoder onto a language model as its "eyes" - the direct next step.
> - **Lesson 6.6 - Video & 3D:** the same encode-into-a-space idea extended across time.
> - **Module 8 (vector search) and Module 15 (bias/moderation):** the vector-database depth and the responsible-use rules named here.

- Dosovitskiy et al., *An Image is Worth 16x16 Words* (ViT, 2020) - [arxiv.org/abs/2010.11929](https://arxiv.org/abs/2010.11929)

- Radford et al., *Learning Transferable Visual Models From Natural Language Supervision* (CLIP, 2021) - [arxiv.org/abs/2103.00020](https://arxiv.org/abs/2103.00020)

- Zhai et al., *Sigmoid Loss for Language Image Pre-Training* (SigLIP, 2023) - [arxiv.org/abs/2303.15343](https://arxiv.org/abs/2303.15343)

- Tschannen et al., *SigLIP 2: Multilingual Vision-Language Encoders* (2025) - [arxiv.org/abs/2502.14786](https://arxiv.org/abs/2502.14786)

- Oquab et al., *DINOv2: Learning Robust Visual Features without Supervision* (2023) - [arxiv.org/abs/2304.07193](https://arxiv.org/abs/2304.07193)

- Hugging Face *transformers* CLIP/SigLIP docs - [huggingface.co/docs/transformers](https://huggingface.co/docs/transformers/model_doc/clip) · OpenCLIP - [github.com/mlfoundations/open_clip](https://github.com/mlfoundations/open_clip)

## Recap

> ✅ **Info**
>
> What we covered
> - **Vision Transformer** - image -> patches -> tokens (+ position) -> transformer -> one embedding.
> - **CLIP** - image + text encoders into one shared space; cosine similarity measures match.
> - **Contrastive learning** - pull matched image-caption pairs together (the diagonal); softmax (CLIP) vs sigmoid (SigLIP).
> - **Zero-shot classification** - nearest class prompt; templates matter; the score is a ranking, not calibrated.
> - **Using CLIP** - search, the SD text encoder (6.2), IP-Adapter (6.3), moderation - and the counting/spatial/text limits.
> - **2026 landscape** - CLIP (baseline), SigLIP 2 (retrieval), DINOv3 (dense), SAM 2; transformers/open_clip/timm.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **ViT & CLIP- When Transformers Learned to See**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-6_4.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-6.4.html` - regenerate this notebook whenever the source HTML is updated.*
